# Building Local OCR Application SmolDocling: A Step-by-Step Guide [Part 2]

Run in order with Python 3.11. Download `app.py` from this same series folder beside the notebook. Install dependencies in an isolated environment:

```bash
python -m pip install streamlit==1.66.1 torch==2.15.0 transformers==5.22.1 docling-core==2.105.0 pillow==13.0.0 requests==2.34.2
```

The pinned full SmolDocling model runs on CPU with real inference. No API key, FlashAttention or paid GPU is required. Model files and the source image download from public URLs. AppTest simulates Streamlit controls while executing the actual model; the separate server cell checks real loopback HTTP.

Run the application interactively from this folder:

```bash
streamlit run app.py --server.address=127.0.0.1 --browser.gatherUsageStats=false
```

Do not expose the development server publicly. The original slide heading is omitted by the model; actual extracted output is preserved.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import importlib.util, inspect, subprocess, sys, tempfile, time
import requests
import hashlib
import io
import threading

from PIL import Image, UnidentifiedImageError
import streamlit as st
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument

MODEL_ID = "docling-project/SmolDocling-256M-preview"
REVISION = "ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8"
MAX_BYTES = 5 * 1024 * 1024
MAX_PIXELS = 10_000_000
DEVICE = "cpu"
print('Python:', sys.version.split()[0])
for package in ['streamlit','torch','transformers','docling-core','pillow']:
    print(package, version(package))
print('Local CPU model:', MODEL_ID, REVISION)

In [ ]:
def decode_image(data):
    if not isinstance(data, bytes) or not 0 < len(data) <= MAX_BYTES:
        raise ValueError("Upload a nonempty image of at most 5 MB.")
    try:
        with Image.open(io.BytesIO(data)) as source:
            if source.format not in {"PNG", "JPEG", "TIFF"}:
                raise ValueError("Use PNG, JPEG or single-page TIFF, not PDF.")
            if source.width * source.height > MAX_PIXELS:
                raise ValueError("The image must contain at most 10 million pixels.")
            if getattr(source, "n_frames", 1) != 1:
                raise ValueError("Upload a single-page image.")
            source.load()
            return source.convert("RGB")
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError) as error:
        raise ValueError("The file is not a supported readable image.") from error
for invalid in [b'', b'%PDF-1.4\nnot an image', b'x' * (MAX_BYTES + 1)]:
    try: decode_image(invalid)
    except ValueError: pass
    else: raise AssertionError('Invalid bytes accepted')
print('Empty, disguised PDF and oversized uploads reject before inference.')

In [ ]:
@st.cache_resource
def load_model():
    torch.set_num_threads(4)
    processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION, token=False)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, revision=REVISION, token=False,
        dtype=torch.float32, attn_implementation="eager"
    ).to(DEVICE).eval()
    return processor, model, threading.Lock()
processor, model, inference_lock = load_model()
assert load_model()[1] is model
assert next(model.parameters()).device.type == 'cpu'
assert next(model.parameters()).dtype == torch.float32 and not model.training
print('One cached model instance:', sum(p.numel() for p in model.parameters()), 'parameters')

In [ ]:
def ocr_pipeline(image):
    if not isinstance(image, Image.Image) or image.width * image.height > MAX_PIXELS:
        raise ValueError("Expected a supported PIL image.")
    processor, model, lock = load_model()
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": "Convert this page to docling."}
    ]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image.convert("RGB")], return_tensors="pt").to(DEVICE)
    with lock, torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=8192, do_sample=False)
    tokens = generated[:, inputs.input_ids.shape[1]:]
    if tokens.shape[1] >= 8192:
        raise RuntimeError("OCR reached its token limit; no incomplete result is returned.")
    tags = processor.batch_decode(tokens, skip_special_tokens=False)[0].lstrip()
    if "<doctag>" not in tags or "</doctag>" not in tags:
        raise RuntimeError("The model did not return a complete document.")
    parsed = DocTagsDocument.from_doctags_and_image_pairs([tags], [image])
    document = DoclingDocument.load_from_doctags(parsed, document_name="Document")
    markdown = document.export_to_markdown()
    if not markdown.strip():
        raise RuntimeError("No text was extracted.")
    return markdown
print('Pipeline reuses the cached processor/model and serializes inference.')

In [ ]:
def main():
    st.set_page_config(page_title="Document OCR Extractor", page_icon="📄", layout="wide")
    st.title("Document OCR Text Extractor")
    st.markdown("Upload a PNG, JPEG or single-page TIFF of at most 5 MB. OCR runs locally on CPU.")
    uploaded = st.file_uploader("Choose an image file", type=["png", "jpg", "jpeg", "tif", "tiff"], key="document", max_upload_size=5)
    if uploaded is None:
        st.session_state.pop("ocr_result", None)
        st.session_state.pop("input_digest", None)
        return
    data = uploaded.getvalue()
    digest = hashlib.sha256(data).hexdigest()
    if st.session_state.get("input_digest") != digest:
        st.session_state.pop("ocr_result", None)
    st.session_state["input_digest"] = digest
    try:
        image = decode_image(data)
    except ValueError as error:
        st.session_state.pop("ocr_result", None)
        st.error(str(error))
        return
    left, right = st.columns(2)
    with left:
        st.subheader("Uploaded Image")
        st.image(image, width="stretch")
    if st.button("Extract text", key="extract"):
        st.session_state.pop("ocr_result", None)
        try:
            with st.spinner("Processing the image..."):
                st.session_state["ocr_result"] = ocr_pipeline(image)
        except (ValueError, RuntimeError, OSError):
            st.error("OCR could not finish. Check the image and local model installation.")
    result = st.session_state.get("ocr_result")
    if result is not None:
        with right:
            st.subheader("Extracted Text")
            st.markdown(result, unsafe_allow_html=False)
        st.download_button("Download Extracted Text", data=result.encode("utf-8"),
                           file_name="extracted_text.md", mime="text/markdown", key="download")
print('Interface defined: validated image upload, explicit extraction, two columns and safe Markdown download.')

In [ ]:
IMAGE_URL = 'https://cdn-images-1.medium.com/max/800/1*Yep_LLJWj4rR8KeUoSCOIg.png'
response = requests.get(IMAGE_URL, timeout=60)
response.raise_for_status()
image_bytes = response.content
image = decode_image(image_bytes)
print('Input image:', image.size)
print('Input SHA256:', hashlib.sha256(image_bytes).hexdigest())

In [ ]:
result = ocr_pipeline(image)
expected = "Here I am writing down every word that I am going to say aloud when I give the presentation. All the text taking up space on the screen is going to make it really difficult for people to listen and to read everything at once, even though they are the same thing. It'd be better if I just went and added a couple key points on the slide and elaborate on them by referring to some flash cards or memorizing what I'm going to say."
assert ' '.join(result.split()) == expected
print(result)

In [ ]:
from streamlit.testing.v1 import AppTest
app_path = Path('app.py').resolve()
assert app_path.is_file(), 'Download the supplied app.py into this series folder.'
at = AppTest.from_file(str(app_path), default_timeout=180).run()
assert not at.exception and len(at.file_uploader) == 1
at.file_uploader[0].set_value(('source-slide.png', image_bytes, 'image/png')).run()
assert not at.exception and len(at.button) == 1
at.button(key='extract').click().run(timeout=180)
assert not at.exception and at.session_state['ocr_result'] == result
assert any(m.value == result for m in at.markdown)
assert at.get('download_button')[0].label == 'Download Extracted Text'
assert 'ocr_result' not in AppTest.from_file(str(app_path)).run().session_state
at.run()
assert at.session_state['ocr_result'] == result
at.file_uploader[0].set_value(('bad.png', b'%PDF-1.4', 'image/png')).run()
assert not at.exception and len(at.error) == 1
assert 'ocr_result' not in at.session_state
assert len(at.get('download_button')) == 0
at.file_uploader[0].set_value(None).run()
assert not at.exception and 'input_digest' not in at.session_state
print('PASS: actual Streamlit AppTest upload/extract callbacks with real OCR, retained result/download, isolated sessions and invalid-upload clearing.')

In [ ]:
# The real local HTTP server is additional to AppTest's in-process UI simulation.
server = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', str(app_path),
    '--server.address=127.0.0.1', '--server.port=18529', '--server.headless=true',
    '--server.fileWatcherType=none', '--browser.gatherUsageStats=false'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for _ in range(120):
        if server.poll() is not None:
            raise RuntimeError('Streamlit stopped: ' + server.stdout.read())
        try:
            health = requests.get('http://127.0.0.1:18529/_stcore/health', timeout=1)
            if health.ok: break
        except requests.RequestException: pass
        time.sleep(0.1)
    else: raise RuntimeError('Local server startup timed out')
    page = requests.get('http://127.0.0.1:18529', timeout=10)
    page.raise_for_status()
    assert health.text == 'ok' and '<html' in page.text.lower()
    print('Real loopback Streamlit health:', health.status_code, health.text)
    print('Real app page:', page.status_code)
finally:
    server.terminate()
    try: server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server.kill(); server.wait(timeout=10)
print('Owned local server stopped.')

In [ ]:
# Additional format/size tests use ordinary image bytes, not model substitutes.
for image_format in ['PNG','JPEG','TIFF']:
    buffer = io.BytesIO(); image.save(buffer, format=image_format)
    decoded = decode_image(buffer.getvalue())
    assert decoded.mode == 'RGB' and decoded.size == image.size
buffer = io.BytesIO(); image.save(buffer, format='TIFF', save_all=True, append_images=[image])
try: decode_image(buffer.getvalue())
except ValueError: pass
else: raise AssertionError('Multiple TIFF pages accepted')
try: ocr_pipeline(None)
except ValueError: pass
else: raise AssertionError('Missing image accepted')
spec = importlib.util.spec_from_file_location('supplied_ocr_app', app_path)
supplied = importlib.util.module_from_spec(spec); spec.loader.exec_module(supplied)
assert supplied.MODEL_ID == MODEL_ID and supplied.REVISION == REVISION
assert 'unsafe_allow_html=False' in app_path.read_text()
assert 'load_model()' in inspect.getsource(supplied.ocr_pipeline)
assert not server.poll() is None
print('PASS: supported image formats, rejected multi-page TIFF/missing input, cached model reuse, safe rendering and completed local server lifecycle.')

## Original source-code and transcript appendix

Nonexecuting original material. Use the maintained cells above.

```text
# Create a virtual environment
python -m venv myenv

# Activate the virtual environment
# On Windows:
myenv\Scripts\activate

# On macOS/Linux:
source myenv/bin/activate
```

```text
streamlit
torch
transformers
pillow
docling
```

```text
# Install packages from requirements.txt
pip install -r requirements.txt
```

```text
import streamlit as st
import torch
from PIL import Image
import io
from transformers import AutoProcessor, AutoModelForVision2Seq
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
```

```text
# Define OCR pipeline function
def ocr_pipeline(image):
    # Load model and processor
    processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")
    model = AutoModelForVision2Seq.from_pretrained(
        "ds4sd/SmolDocling-256M-preview",
        torch_dtype=torch.bfloat16,
    ).to(DEVICE)
    
    # Prepare input message
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Convert this page to docling."}
            ]
        },
    ]
    
    # Process input
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(DEVICE)
    
    # Generate output
    generated_ids = model.generate(**inputs, max_new_tokens=8192)
    trimmed_generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    
    # Decode output
    doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()
    
    # Create Docling document
    doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
    doc = DoclingDocument(name="Document")
    doc.load_from_doctags(doctags_doc)
    
    return doc.export_to_markdown()
```

```text
# Set page configuration
st.set_page_config(
    page_title="Document OCR Extractor",
    page_icon="📄",
    layout="wide"
)

# Determine device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Cache model loading to improve performance
@st.cache_resource
def load_model():
    processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")
    model = AutoModelForVision2Seq.from_pretrained(
        "ds4sd/SmolDocling-256M-preview",
        torch_dtype=torch.bfloat16,
    ).to(DEVICE)
    return processor, model
```

```text
# Main application
def main():
    # Application header
    st.title("Document OCR Text Extractor")
    st.markdown("Upload an image containing text to extract its content.")
    
    # File uploader
    uploaded_file = st.file_uploader("Choose an image file", type=["jpg", "jpeg", "png", "tiff", "pdf"])
    
    # Process the uploaded image
    if uploaded_file is not None:
        # Display a spinner during processing
        with st.spinner("Processing the image..."):
            # Read the image
            image_bytes = uploaded_file.read()
            image = Image.open(io.BytesIO(image_bytes))
            
            # Display the uploaded image
            col1, col2 = st.columns(2)
            with col1:
                st.subheader("Uploaded Image")
                st.image(image, use_column_width=True)
            
            # Extract text using OCR pipeline
            result = ocr_pipeline(image)
            
            # Display the extracted text
            with col2:
                st.subheader("Extracted Text")
                st.markdown(result, unsafe_allow_html=True)
            
            # Provide download button for the text
            st.download_button(
                label="Download Extracted Text",
                data=result,
                file_name="extracted_text.md",
                mime="text/markdown"
            )

if __name__ == "__main__":
    main()
```

```text
streamlit run app.py
```

```text
You can find the codes and data used in this article in this GitHub Repo.
```

